# Tensile and printability data — all 12 formulations at cycles 0 and 1 (2026-11-10)

Newest Drive snapshot, `raw_data/11_04_26/` (pulled by `scripts/sync_drive.py`): the four **TPU formulations at cycle 1** arrived on 4 Nov (20 specimens), so every formulation now has cycle 0 and cycle 1. Extends `data_processing_v5.ipynb`; same sections, section 9 replaced by the **Option 1 / Option 2** comparison agreed with Anupam on 4 Nov (`meeting_notes/2026-11-04_doe_options_anupam.md`).

Logic in `utils/instron.py`, `utils/analysis.py`, `utils/doe.py`; figures in `utils/plots.py`. Run in the `ml_gp_env` kernel. **Nothing is written by default.**

## Key points

* **Coverage:** 24 runs (12 formulations × cycles 0 and 1), 102 specimens analysed (104 received; f11_s3 and the original f08 set aside). Nothing at cycles 2–3.
* **One loop acts differently on each polymer.** Polymer averages, cycle 1 ÷ cycle 0: UTS PP ×1.07, HDPE ×1.32, TPU ×1.37; modulus ×0.98, ×1.15, **×2.38**; strain ×1.11, ×1.22, **×0.42**; toughness ×1.18, ×1.75, ×0.55. PP and HDPE hold or gain; TPU stiffens and loses ductility. Polymer × cycle: modulus p = 0.007, strain p = 0.04.
* **The all-formulation average hides this:** UTS +24 % (95 % CI −1 to +55, p = 0.06), modulus +39 % (+2 to +90, p = 0.04), strain −17 %, toughness +4 % (12 pairs).
* **The temperature lead from the first 8 formulations does not hold across 12** (p ≥ 0.49). Diameter still does: the modulus change follows the change in filament diameter (r = −0.65, p = 0.02); within a run thicker specimens read lower UTS (r = −0.45).
* **Two TPU runs split in two:** F05R1 and F07R1 have 2 stiff-brittle and 3 soft-ductile specimens each (CV 65–89 % for modulus, strain, toughness). Kept, flagged, to ask the lab. F07R1: specimen 1 missing, two files without a diameter.
* **Cycle 0 results stand:** polymer first, then PAL; polymer × PAL for modulus, strain, toughness; temperature not resolved.
* **Kernel check:** LVGP vs `lvgp_mixed` on the 24 runs, scored by paired leave-one-out NLPD (the proper score: mean and uncertainty together). Leave-one-run-out: LVGP clearly better for UTS (20 of 24 folds), ties elsewhere; leave-one-formulation-out: ties, both over-confident (≈60 % within ±2 SD); log evidence higher for the LVGP on all four. The deck keeps the LVGP. Details in `modeling/lvgp_recycling.ipynb` §6.
* **Printability:** unchanged from v5 (warping sheet has no new rows).
* **DoE (section 9):** two options for cycles 2–3. **Option 1** continues the 12 formulations (24 runs, 0 new formulations, 24 cycle operations, 120 specimens, direct pairs; L2 all / within 0.022 / 0.053, |r| 0.30). **Option 2** adds 6 new LHS configurations at cycle 2 and 6 at cycle 3 (12 runs, 12 new formulations, 30 cycle operations, 60 specimens; 0.036 / 0.048, |r| 0.06). Anupam decides on manufacturing workload. Plan: `doe/doe_plan_v6_option2.xlsx`.

**Caution:** cycle is confounded with test session, and batches differ more than replicates (lack of fit p < 0.001), so treat the recycle numbers as a first look.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "utils").exists() and (ROOT.parent / "utils").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import config, io, doe, plots

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
plt.rcParams["figure.dpi"] = 110
p = config.paths(ROOT)

In [ ]:
from utils import instron, analysis as A

RAW = p.root / "raw_data" / "11_04_26"
design = pd.read_csv(p.doe / "doe_formulations.csv")
cat = instron.catalogue(io.extract_zip(RAW))
print((RAW / "CHANGES.txt").read_text()[:200])
cat.groupby("recycled_x").agg(specimens=("specimen_id", "size"), formulations=("formulation", "nunique"))

---
## 1 — Parse, set aside, screen

Same two set-asides as v4. No recycled specimen is removed; the pooled screen, now per run (formulation at one cycle), flags **f04r1_s4 on all four responses**: worth a question to the lab.

In [ ]:
SAVE_CSV = False                      # True -> write the per-specimen table to processed_data/tensile_doe/
CSV_NAME = "tensile_nov_summary"

curves, raw = instron.load_delivery(cat)
summary = instron.attach_doe(raw, design)
DROP = {"f11_s3": "failed test: no load recorded (UTS 0.12 MPa, modulus fit R² 0.001)",
        "f08_s1": "superseded by the f08 remake (f08-new): 119 % strain vs 7 % for the 3 remade specimens"}
kept, dropped = A.drop_specimens(summary, DROP)
kept0, kept1 = kept[kept["recycled_x"] == 0], kept[kept["recycled_x"] == 1]
if SAVE_CSV:
    summary.to_csv(io.save_path(p.tensile_doe, CSV_NAME, ext=".csv"), index=False)
print(f"{len(kept)} specimens kept: {len(kept0)} at cycle 0, {len(kept1)} at cycle 1")
dropped

In [ ]:
flags = A.pooled_outliers(kept)          # pooled log residuals from the run median, IQR fences
flags[flags["n_flags"] > 0]

In [ ]:
cv = A.cv_table(kept)
same8 = cv[cv["formulation"].isin(kept1["formulation"])]
print("median CV %, all runs:"); print(cv.groupby("cycle")[list(A.RESPONSES)].median().round(1))
print("median CV %, the 8 formulations with both cycles:"); print(same8.groupby("cycle")[list(A.RESPONSES)].median().round(1))
cv.round(2)

---
## 2 — Curves and properties

Cycle 0 figures as in v4; the properties-vs-factors figure now also shows cycle 1 (hollow).

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_curves_cycle0"

plots.tensile_curves_by_polymer(curves, kept0, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_props_cycle0"

plots.props_by_formulation_log(kept0, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_props_vs_factors"

plots.props_vs_factors_log(kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

---
## 3 — The recycle step: the same formulation at cycles 0 and 1

The formulation is the unit: 8 pairs. Ratio > 1 means higher after recycling.

**Gain or loss?** Each thin line is one formulation; the black line is the average of the 8, with its 95 % CI. Cycle 0 = first extrusion (lab's 0x), cycle 1 = one loop (1x).

In [ ]:
pairs = A.cycle_pairs(kept)
effect = A.cycle_effect(pairs)
means = A.cycle_means(pairs)
means.round(3)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_cycle_gain_loss"

plots.cycle_gain_loss(pairs, effect, means, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_curves_by_cycle"

plots.curves_by_cycle(curves, kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
effect.round(3)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_cycle_forest"

plots.cycle_ratio_forest(pairs, effect, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

**Is the behaviour the same?** Slopes per cycle on the 8 formulations with both, % per +10 PAL points and per +10 °C, each adjusted for the other input; `p_change` tests whether the slope changed. PAL keeps its direction; temperature reverses for UTS, strain and toughness. Exploratory.

In [ ]:
A.cycle_trend_table(kept).round(3)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_cycle_trends"

plots.cycle_trends(kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

Does the change depend on the factors? Temperature and diameter stand out. **Exploratory:** 8 points, several tests, chosen after seeing the data. `p_loo` is the largest p after removing any one formulation.

In [ ]:
A.cycle_interactions(pairs).round(3)

In [ ]:
A.cycle_vs_factors(pairs, kept).round(3)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_cycle_vs_temperature"

plots.cycle_ratio_vs_temperature(pairs, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

Thicker specimens of one run read lower UTS and modulus. A slope of −2 would be pure area (stress ∝ 1/d²), so this could be measurement of the cross-section, or voids in thick filament.

In [ ]:
A.diameter_effect(kept).round(3)

In [ ]:
# Sensitivity: without the flagged specimen, and without F01
print(A.cycle_effect(A.cycle_pairs(kept[kept["specimen_id"] != "f04r1_s4"])).query("group == 'all'")[["response", "ratio", "lo", "hi", "p"]].round(3))
print(A.cycle_effect(pairs[pairs["formulation"] != 1]).query("group == 'all'")[["response", "ratio", "lo", "hi", "p"]].round(3))

---
## 4 — Pair plot and correlation between the responses

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_pairs"

plots.pair_plot_specimens(kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
corrs = A.response_correlation(kept, polymers=config.DOE.polymers)
plots.response_correlation(corrs)

---
## 5 — ANOVA

Pooling all 20 run means would treat the 8 repeated formulations as independent. So: **polymer, PAL and temperature are tested on the 12 cycle-0 means** and **the recycle step on the 8 pairs**; a mixed model is a cross-check.

In [ ]:
A.lack_of_fit(kept, rhs="C(polymer) * pal + temp + cycle").round(4)

In [ ]:
an0 = A.anova_means(kept0, rhs="C(polymer) * pal + temp")   # one PAL slope per polymer
x = np.log(pairs.assign(_r=pairs["ratio"]).pivot(index="formulation", columns="response", values="_r"))
t = x.mean() / (x.std(ddof=1) / np.sqrt(len(x)))
step = pd.DataFrame({"response": t.index, "effect": "recycle step (cycle 0 → 1)", "df": 1, "F": t.to_numpy() ** 2,
                     "p": effect.query("group == 'all'").set_index("response").loc[t.index, "p"].to_numpy(),
                     "partial_eta2": (t ** 2 / (t ** 2 + len(x) - 1)).to_numpy()})
an = pd.concat([an0, step], ignore_index=True)
an.round(4)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_anova"

plots.anova_effects(an, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
A.interaction_tests(kept0).round(4)

In [ ]:
print("PAL effect, % per +10 PAL points, cycle 0"); print(A.pal_effect_by_polymer(kept0).round(1))
print("cycle 1"); print(A.pal_effect_by_polymer(kept1).round(1))

In [ ]:
A.anova_means(kept1, rhs="C(polymer) + pal + temp").round(4)   # the 8 cycle-1 means

In [ ]:
import statsmodels.formula.api as smf
import warnings
rows = []
for r in A.RESPONSES:                    # cross-check of the recycle step: random intercept per formulation
    fm = A._formulation_means(kept, r)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m = smf.mixedlm("ly ~ C(polymer) * pal + temp + cycle", fm, groups=fm["formulation"]).fit(reml=True)
    rows.append({"response": r, "ratio": np.exp(m.params["cycle"]), "p": m.pvalues["cycle"]})
pd.DataFrame(rows).round(3)

---
## 6 — Partial dependence and Sobol indices with the recycle cycle as an input

GP as in v4 with the cycle as one more input. Each polymer is drawn only at cycles it has data for. Sobol indices use PP and HDPE only: TPU has no recycled data, and is not extrapolated. About 5 min.

In [ ]:
pds = {r: A.gp_partial_dependence_cycles(kept, r, config.DOE.polymers) for r in A.RESPONSES}
{r: d.attrs["kernel"] for r, d in pds.items()}

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_partial_dependence"

plots.partial_dependence_cycles(pds, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
sob = pd.concat([A.gp_sobol_cycle(kept, r) for r in A.RESPONSES], ignore_index=True)
(sob.assign(order=sob["order"].astype(str)).pivot_table(index=["order", "term"], columns="response", values="median", sort=False)
 [list(A.RESPONSES)].round(3))

---
## 7 — Printability: corner warping

From the lab's `Sample Warping.xlsx`. Counts of 4, 8 and 12 suggest 4 corners on 1, 2 or 3 samples (**to confirm**). F09 is missing, F04 has one usable sample, and **the sheet does not say which cycle**, so it is compared with cycle 0 here. Readings are in steps of 0.05–0.1 cm.

In [ ]:
warp = A.load_warping(RAW / "extracted" / "printability" / "Sample Warping.xlsx", design)
A.warping_summary(warp).round(3)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "warping_by_formulation"

plots.warping_by_formulation(warp, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
wt = A.warping_tests(warp, kept)
print("between formulations (Kruskal-Wallis):", {k: round(float(v), 6) for k, v in wt["between"].items()})
print(wt["anova"].round(4))
wt["tensile"].round(3)

**Does adding PAL or changing temperature move warping?** Straight-line slopes in cm, and a curvature test for temperature. The LVGP surfaces and its leave-one-out check are in `modeling/lvgp_recycling.ipynb`.

In [ ]:
A.warping_slopes(warp).round(3)

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
ws = A.warping_summary(warp).rename(columns={"polymer_system": "polymer", "pal_pct": "pal", "extr_temp_C": "temp"})
ws["t"] = ws["temp"] - ws["temp"].mean()
straight, curved = smf.ols("mean ~ C(polymer) + pal + t", ws).fit(), smf.ols("mean ~ C(polymer) + pal + t + I(t ** 2)", ws).fit()
print("lowest warping at about", round(ws["temp"].mean() - curved.params["t"] / (2 * curved.params["I(t ** 2)"])), "degC")
anova_lm(straight, curved).round(4)

Warping rises with UTS and modulus across formulations (Spearman 0.75, 0.80), but that is mostly the polymer: TPU is soft and warps least. With 4 formulations per polymer it cannot be separated within a polymer yet.

---
## 8 — Trade-offs

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_tradeoffs"

plots.tradeoffs(kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

---
## 9 — Cycles 2–3: Option 1 vs Option 2

Agreed with Anupam on 4 Nov (`meeting_notes/2026-11-04_doe_options_anupam.md`): compare two routes on actual manufacturing workload. **Option 1** continues the existing 12 formulations through cycles 2 and 3. **Option 2** adds **6 new configurations at cycle 2 and 6 at cycle 3** (2 per polymer each), placed by the per-polymer LHS conditioned on all 24 fabricated runs. Takes a few minutes.

In [ ]:
fixed = doe.existing_runs(design, cat)
fixed.groupby(["polymer_system", "recycle_cycle"]).size().unstack(fill_value=0)

In [ ]:
cols = ["polymer_system", "PAL_pct", "extrusion_temp_C", "recycle_cycle"]
option1 = pd.concat([design.assign(recycle_cycle=c) for c in range(4)], ignore_index=True)[cols]     # the same 12 at cycles 0-3
print("Option 1, runs per polymer x cycle:"); print(option1.groupby(["polymer_system", "recycle_cycle"]).size().unstack(fill_value=0))
pd.Series(doe.design_quality(option1)).to_frame("Option 1 (48 runs)")

In [ ]:
N_PER_POLYMER = 4          # 12 new runs; the even allocation puts them at cycles 2 and 3, two each per polymer
seeds = doe.conditioned_seeds(fixed, range(10), n_per_polymer=N_PER_POLYMER)
seeds.sort_values(["poly_fill_dist", "all_min_dist"], ascending=[True, False]).round(3)

In [ ]:
base = doe.compare_conditioned_baselines(fixed, n_seeds=10, n_new=3 * N_PER_POLYMER)
base.groupby("method").mean(numeric_only=True).round(3).T

In [ ]:
SAVE_PLAN = False                     # True -> write doe/<PLAN_NAME>.xlsx and .csv (already written once, 2026-11-10)
PLAN_NAME = "doe_plan_v6_option2"     # Option 2: 6 new configurations at cycle 2 + 6 at cycle 3
PICK_SEED = 1                         # seeds tie on discrepancy; this one has the largest closest-pair distance

new, q = doe.conditioned_sliced_lhs(fixed, N_PER_POLYMER, seed=PICK_SEED)
if SAVE_PLAN:
    doe.write_plan_workbook(new, fixed, p.doe / f"{PLAN_NAME}.xlsx")
    new.assign(tensile_replicates=5).to_csv(p.doe / f"{PLAN_NAME}.csv", index=False)
option2 = pd.concat([fixed[cols], new[cols]], ignore_index=True)
print(pd.DataFrame({"24 fabricated": doe.design_quality(fixed[cols]), "Option 1 (48)": doe.design_quality(option1), "Option 2 (36)": doe.design_quality(option2)}))
new

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "doe_v6_option2_design_space"

plots.doe_design_space(new, existing=fixed, color_by="cycle", save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "doe_v6_option2_input_correlation"

plots.doe_correlation(option2, title="Correlation between inputs: 24 fabricated + 12 new runs (Option 2)", save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

**Balance under Option 2.** Runs per polymer × cycle before and after, and each polymer's PAL and temperature settings. Even across polymers (12 each) and composition (8 settings per polymer); cycles 2–3 get 6 runs each against 12 at cycles 0–1. Option 1 would give 12 at every cycle.

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "doe_v6_option2_balance"

print(option2.groupby(["polymer_system", "recycle_cycle"]).size().unstack(fill_value=0))
plots.design_balance(new, fixed, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

**Workload, the way Anupam asked for it.** Both options at 5 replicates per run. Option 2 saves tests but each new formulation has to be compounded and taken through cycles 0 and 1 before it reaches cycle 2 or 3, so it needs more recycle-cycle operations (6 × 2 + 6 × 3 = 30) than Option 1 (one more loop for each of 24 runs). If the cycle-0 and cycle-1 material of the new formulations is also tested, Option 2 yields 24 extra runs (120 specimens) at the lower cycles. Option 1 gives a direct before/after pair for every formulation at every cycle, which matters because batches differ more than replicates; Option 2 doubles the composition settings per polymer and makes the inputs independent.

In [ ]:
n_new1 = len(option1) - len(fixed)
pd.DataFrame({"Option 1: continue the 12": {"new runs to test": n_new1, "new formulations to compound": 0, "recycle-cycle operations": n_new1, "tensile specimens": n_new1 * 5,
                                             "PAL-T settings per polymer": 4, "direct pairs at cycles 2-3": "yes, 12 each"},
              "Option 2: 6 + 6 new": {"new runs to test": len(new), "new formulations to compound": len(new), "recycle-cycle operations": int(new["recycle_cycle"].sum()), "tensile specimens": len(new) * 5,
                                      "PAL-T settings per polymer": 8, "direct pairs at cycles 2-3": "no (model)"}})

In [ ]:
pd.DataFrame({"24 now": doe.design_quality(fixed[cols]), "Option 1 (48)": doe.design_quality(option1), "Option 2 (36)": doe.design_quality(option2)})   # the fill distance favours the larger repeated plan

---
## Open questions for the lab

1. TPU F05 and F07 at cycle 1: two material states among the five specimens (2 stiff and brittle, 3 soft and ductile). What differed?
2. F07R1: specimen 1 missing, two files without a diameter. F04 at cycle 1 (f04r1_s4) and F01 at cycle 0: anything different in the batch or test session?
3. How is the diameter measured, and is the second extrusion at the same temperature as the first?
4. Warping: which cycle, what is one reading, is F09 coming? Measure it on every run from now on.
5. Option 1 or Option 2 for cycles 2–3: Anupam's manufacturing-time estimate for both.